In [3]:
%pip install spotipy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import time

# Client ID set-up for API
client_id = 'd71f5ebb2095498d998db9f448b8b2b1'
client_secret = '5dcc3dd468b64967991fcb2343bfef4d'

client_credentials_manager = SpotifyClientCredentials(client_id, client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
#example code: Dua Lipa
result = sp.search("Dua Lipa")
track = result['tracks']['items'][0]

artist = sp.artist(track["artists"][0]["external_urls"]["spotify"])
print("artist genres:", artist["genres"])

album = sp.album(track["album"]["external_urls"]["spotify"])
print("album genres:", album["genres"])
print("album release-date:", album["release_date"])

artist genres: ['dance pop', 'pop', 'uk pop']
album genres: []
album release-date: 2020-03-27


## Set Up 'get_genre' Function

In [ ]:
def get_genre(artist):
    result = sp.search(artist)
    track = result['tracks']['items'][0]
    artist = sp.artist(track["artists"][0]["external_urls"]["spotify"])
    
    genre = artist['genres']
    return genre

In [ ]:
get_genre('Dua Lipa')

Out[4]: ['dance pop', 'pop', 'uk pop']

## Load CSV and add Genre

In [ ]:
import pandas as pd
df = pd.read_csv('/Workspace/Users/carrie.oh@curinos.com/artists.csv')
df['artist_genre'] = str("")
df

,year,month,week,rank,track title,artist name,artist_genre
0,2021,11,1,1,NaN,Drake,
1,2021,11,1,2,NaN,Taylor Swift,
2,2021,11,1,3,NaN,Kanye West,
3,2021,11,1,4,NaN,Juice WRLD,
4,2021,11,1,5,NaN,Ed Sheeran,
...,...,...,...,...,...,...,...
4995,2023,11,1,196,NaN,System Of A Down,
4996,2023,11,1,197,NaN,Grupo Firme,
4997,2023,11,1,198,NaN,Justin Timberlake,
4998,2023,11,1,199,NaN,Foo Fighters,


In [ ]:
#creating artist:genre dictionary for shorter runtime
artist_dict = {}
for i in df['artist name']:
    artist_dict[i] = ""
#artist_dict

In [ ]:
len(artist_dict.keys())

Out[7]: 394

In [ ]:
#pull genre using spotipy
for i in artist_dict.keys():
    artist_dict[i] = get_genre(i)
artist_dict

Out[25]: {'Drake': ['canadian hip hop', 'canadian pop', 'hip hop', 'pop rap', 'rap'],
 'Taylor Swift': ['pop'],
 'Kanye West': ['chicago rap', 'hip hop', 'rap'],
 'Juice WRLD': ['chicago rap', 'melodic rap', 'rap'],
 'Ed Sheeran': ['pop', 'singer-songwriter pop', 'uk pop'],
 'The Weeknd': ['canadian contemporary r&b', 'canadian pop', 'pop'],
 'Doja Cat': ['dance pop', 'pop'],
 'Bad Bunny': ['reggaeton', 'trap latino', 'urbano latino'],
 'Polo G': ['chicago rap', 'rap'],
 'Lil Baby': ['atl hip hop', 'atl trap', 'rap', 'trap'],
 'Justin Bieber': ['canadian pop', 'pop'],
 'Lil Uzi Vert': ['hip hop', 'melodic rap', 'philly rap', 'rage rap', 'rap'],
 'J. Cole': ['conscious hip hop', 'hip hop', 'north carolina hip hop', 'rap'],
 'Billie Eilish': ['indie hip hop'],
 'Travis Scott': ['hip hop', 'rap', 'slap house'],
 'YoungBoy Never Broke Again': ['baton rouge rap', 'rap'],
 'Ariana Grande': ['pop'],
 'Young Thug': ['atl hip hop',
  'atl trap',
  'gangster rap',
  'melodic rap',
  'rap',
  'tr

In [ ]:
#add genre to df
for i in range(len(df)):
    df['artist_genre'].iloc[i] = artist_dict[df['artist name'].iloc[i]]

/databricks/python/lib/python3.9/site-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)


In [ ]:
display(df)

year,month,week,rank,track title,artist name,artist_genre
2021,11,1,1,null,Drake,"List(canadian hip hop, canadian pop, hip hop, pop rap, rap)"
2021,11,1,2,null,Taylor Swift,List(pop)
2021,11,1,3,null,Kanye West,"List(chicago rap, hip hop, rap)"
2021,11,1,4,null,Juice WRLD,"List(chicago rap, melodic rap, rap)"
2021,11,1,5,null,Ed Sheeran,"List(pop, singer-songwriter pop, uk pop)"
2021,11,1,6,null,The Weeknd,"List(canadian contemporary r&b, canadian pop, pop)"
2021,11,1,7,null,Doja Cat,"List(dance pop, pop)"
2021,11,1,8,null,Bad Bunny,"List(reggaeton, trap latino, urbano latino)"
2021,11,1,9,null,Polo G,"List(chicago rap, rap)"
2021,11,1,10,null,Lil Baby,"List(atl hip hop, atl trap, rap, trap)"


In [ ]:
from datetime import date

today = date.today()

In [ ]:
df.to_csv(f'{today} artists_w_genre.csv')

## Create Genre Score


In [ ]:
#### If need to load csv made above ####
#df = pd.read_csv("/Workspace/Users/carrie.oh@curinos.com/2023-12-03 Artists_w_genre.csv")
#df

In [ ]:
###### IF artist_genre is saved as str not list #####
"""
import ast
for i in range(len(df)):
    string = df.artist_genre[i]
    df.artist_genre[i] = ast.literal_eval(string)

df
"""

Out[78]: '\nimport ast\nfor i in range(len(df)):\n    string = df.artist_genre[i]\n    df.artist_genre[i] = ast.literal_eval(string)\n\ndf\n'

In [ ]:
import itertools
genre_list = []
for i in range(len(df)):
    genre_list.append(df.artist_genre[i])

genre_list = list(itertools.chain.from_iterable(genre_list))
genre_list

Out[48]: ['canadian hip hop',
 'canadian pop',
 'hip hop',
 'pop rap',
 'rap',
 'pop',
 'chicago rap',
 'hip hop',
 'rap',
 'chicago rap',
 'melodic rap',
 'rap',
 'pop',
 'singer-songwriter pop',
 'uk pop',
 'canadian contemporary r&b',
 'canadian pop',
 'pop',
 'dance pop',
 'pop',
 'reggaeton',
 'trap latino',
 'urbano latino',
 'chicago rap',
 'rap',
 'atl hip hop',
 'atl trap',
 'rap',
 'trap',
 'canadian pop',
 'pop',
 'hip hop',
 'melodic rap',
 'philly rap',
 'rage rap',
 'rap',
 'conscious hip hop',
 'hip hop',
 'north carolina hip hop',
 'rap',
 'indie hip hop',
 'hip hop',
 'rap',
 'slap house',
 'baton rouge rap',
 'rap',
 'pop',
 'atl hip hop',
 'atl trap',
 'gangster rap',
 'melodic rap',
 'rap',
 'trap',
 'british soul',
 'pop',
 'pop soul',
 'uk pop',
 'atl hip hop',
 'hip hop',
 'rap',
 'southern hip hop',
 'trap',
 'detroit hip hop',
 'hip hop',
 'rap',
 'cloud rap',
 'dark trap',
 'new orleans rap',
 'underground hip hop',
 'atl hip hop',
 'hip hop',
 'rap',
 'hip ho

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   year          5000 non-null   int64  
 1   month         5000 non-null   int64  
 2   week          5000 non-null   int64  
 3   rank          5000 non-null   int64  
 4   track title   0 non-null      float64
 5   artist name   5000 non-null   object 
 6   artist_genre  5000 non-null   object 
dtypes: float64(1), int64(4), object(2)
memory usage: 273.6+ KB


In [ ]:
key = genre_list
value = [0]*len(key)
key_value = zip(key,value)
genre_rank = dict(key_value)

for i in range(len(df)):
    rank_inverse = 1/df.iloc[i]['rank']
    for genre in df.artist_genre[i]:
        genre_rank[genre] += rank_inverse

genre_rank_sorted = dict(sorted(genre_rank.items(), key=lambda item: item[1], reverse=True))


In [ ]:
genre_rank_sorted

Out[76]: {'rap': 64.89774784006376,
 'pop': 45.27196555621168,
 'hip hop': 41.87079420966266,
 'pop rap': 20.06075270993361,
 'canadian pop': 19.936482705439253,
 'canadian hip hop': 14.53780997247489,
 'trap': 13.646922865974455,
 'trap latino': 11.914556989222483,
 'urbano latino': 11.362530363598738,
 'chicago rap': 11.300946371270559,
 'melodic rap': 11.300841521062615,
 'atl hip hop': 11.247365896464979,
 'reggaeton': 11.234494160718308,
 'contemporary country': 7.176189039835229,
 'r&b': 5.686872267141955,
 'canadian contemporary r&b': 4.586181849622131,
 'southern hip hop': 4.518461948800804,
 'rock': 4.385289764348169,
 'dance pop': 3.826219901899467,
 'conscious hip hop': 3.6211093638587566,
 'sad sierreno': 3.4162512018257756,
 'corridos tumbados': 3.1691228103697977,
 'sierreno': 3.0838064129078138,
 'modern rock': 2.979793919105793,
 'corrido': 2.978059098139599,
 'atl trap': 2.9240069390019836,
 'rage rap': 2.4382490138866353,
 'new orleans rap': 2.334641095169594,
 'pov: 